In [1]:
%load_ext autoreload
%autoreload 2

import sys, os, json
dir = os.getcwd()

ext = ['', '/..', '/../..', '/../../src/models', '/../../src/nlp', '/../../src/synth']
sys.path += [dir + i for i in ext]

In [2]:
from eval import *
from state import *
from expectation import *

In [3]:
from api.football import *

In [4]:
# Get folder path
folder_path = os.path.join('data')

demos = []
for i in os.listdir(folder_path):
    if i.startswith('demonstration0'): demos += [i]
    
demos.sort()

print(demos)

['demonstration0']


In [5]:
import json
from scene import Scene
from api.objects.registry import REGISTRY

scenes = {}

i = 0
for d in demos:
    if not i: # i=0 for first demo
        print('initial demo (w/ language)')
    else:
        print(f'helper demo {i}')

    folder = os.path.join(folder_path, d, 'json_segments')
    dir_list = [i for i in os.listdir(folder) if i.endswith('.json')]
    dir_list.sort()

    # print(dir_list)

    j = 0
    for f in dir_list:

        file = os.path.join(folder, f)

        # print(file)

        with open(file) as f:
            data = json.load(f)
        
        # for t in data['scene']:
        #     print(t)
        
        if not i:
            scenes[j] = Scene.from_dict(data['scene'], REGISTRY)
        else:
            try:
                scenes[j].add_demo(Scene.from_dict(data['scene'], REGISTRY))
            except KeyError:
                print(f"Attempted to access part {j} of a demonstration. Either non-existant or not in the original demo.")

        print(f'part {j} -> {scenes[j]}')

        j += 1
    

    i += 1
    print()

initial demo (w/ language)
part 0 -> <scene.Scene object at 0x133fc9d50>



In [6]:
objects = {obj.id: obj for obj in scenes[0].allObjects}

for id in objects.keys():
    print(id)

corner1
corner2
corner3
corner4
coach
opponent_A
opponent_B
opponent_C
opponent_D
teammate
ball
goal
goal_leftpost
goal_rightpost
target


In [7]:
teammate_has_ball = HasBallPosession(objects['teammate'])
coach_has_ball = HasBallPosession(objects['coach'])

In [8]:
from expectation import *

teammate_received_ball = DidHappen([
    coach_has_ball,
    (coach_has_ball, False),
    teammate_has_ball
])

In [9]:
class PlayerAheadOfOpponents(State):

    def __init__(self, object):
        super().__init__(f'{object.id} is ahead of opponents')
        self.id = f'PlayerAheadOfOpponents:{object.id}'
        self.object = object

    def check(self, scene, objects, t) -> bool:
        opponents = [objects['opponent_A'], objects['opponent_B'], objects['opponent_C'], objects['opponent_D']]
        for opponent in opponents:
            if objects[self.object.id]._position[t].y < opponent._position[t].y:
                return False
        return True

In [10]:
coach_received_ball = DidHappen([
    teammate_has_ball,
    (teammate_has_ball, False),
    coach_has_ball
])

teammate_ahead_opponents = PlayerAheadOfOpponents(objects['teammate'])

In [11]:
from eval import Eval
eval = Eval(scenes)

In [12]:
teammate_forward_pass = Eventually(
    before=[
        teammate_ahead_opponents
    ],
    after=[
        teammate_has_ball
    ]
)

teammate_moved_past_defending_line = DidHappen(
    teammate_ahead_opponents
)

In [13]:
eval.sub([
    teammate_has_ball,
    coach_has_ball,
    teammate_ahead_opponents
])

eval.verify([
    teammate_received_ball,
    coach_received_ball,
    teammate_forward_pass
])

In [14]:
score = eval.run()

[1, 1, 1]
Score: 3/3


In [15]:
eval.timeline

[(coach has ball posession, True),
 (coach has ball posession, False),
 (teammate has ball posession, True),
 (teammate has ball posession, False),
 (coach has ball posession, True),
 (teammate is ahead of opponents, True),
 (coach has ball posession, False),
 (teammate has ball posession, True)]